In [ ]:
# Audio-apparaten controleren

import sounddevice as sd

# Toon de lijst met alle beschikbare audio-apparaten
print("Lijst met gedetecteerde apparaten:")
print(sd.query_devices())

In [ ]:
# Cel 1
import sounddevice as sd
import soundfile as sf
import os
import numpy as np

# Audioconfiguratie
Rate = 44100       # Samplefrequentie (CD-kwaliteit, legt vast tot ~22kHz)
duration = 5    # Opnameduur in seconden
Channels = 1       # Aantal kanalen (1=mono, 2=stereo)
output_file = "test.wav"  # Naam van het doelbestand

# Controle op bestaand bestand
doorgaan = True
if os.path.exists(output_file):
    keuze = input(f"⚠️ {output_file} bestaat al. [Enter] = overschrijven, [q] = stop: ").lower()
    if keuze == 'q':
        print("🛑 Opname afgebroken.")
        doorgaan = False

if doorgaan:
    print("🎤 Start opname...")
    recording = sd.rec(
        int(duration * Rate), 
        samplerate = Rate, 
        channels = Channels, 
        dtype = 'float32',
        device = None     # Pakt automatisch het apparaat met het > pijltje
    )
    sd.wait()
    print("🛑 Opname gestopt.")

    # Opslaan
    sf.write(output_file, recording, Rate)
    print(f"💾 Opname succesvol opgeslagen als {output_file}")

    plot_color = 'green'
    # Clipping-detectie (kijkt naar absolute pieken boven 0.99)
    if np.any(np.abs(recording) >= 0.99):
        print("⚠️ Waarschuwing: Clipping gedetecteerd! Geluid is mogelijk vervormd.")
        plot_color = 'red'
    else:
        print("✅ Geen clipping gedetecteerd.")

# extra visuele controle
import matplotlib.pyplot as plt

time = np.linspace(0., duration, len(recording))


recording_scale = recording * 32768  # omzetten naar een 16-bit audio scale
# Grafiek plotten
plt.figure(figsize=(10, 6))
plt.plot(time, recording_scale, color=plot_color, alpha=0.7)

# Instellen van de assen voor 16-bit audio
plt.ylim(-32768, 32768)

plt.title('Geluidssignaal over de tijd')
plt.xlabel('t (s)')
plt.ylabel('A')
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
import pandas as pd

# Inlezen van het .wav bestand
file_path = 'test.wav'
sampling_rate, audio_data = wavfile.read(file_path)

# Tijd-array maken
duration = len(audio_data) / sampling_rate
time = np.linspace(0., duration, len(recording))

# Grafiek plotten
plt.figure(figsize=(10, 6))
plt.plot(time, audio_data, color='blue', alpha=0.7)

# Instellen van de assen voor 16-bit audio
plt.ylim(-32768, 32768)

plt.title('Geluidssignaal over de tijd')
plt.xlabel('t (s)')
plt.ylabel('A')
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd              # Import pandas voor werken met tabellen (DataFrames)
pd.set_option('display.max_rows', None)  # Zorgt dat alle rijen zichtbaar zijn bij printen

df = pd.DataFrame(audio_data)   # Zet ruwe audio_data om naar een DataFrame

window_size = int(0.02*sampling_rate)  
# Bepaalt de grootte van het venster (20 ms)
# sampling_rate = aantal samples per seconde
# 0.02 * sampling_rate = aantal samples in 20 milliseconden

data = rolling_mean = np.convolve(audio_data, np.ones(window_size)/window_size, mode='valid')
# Berekent een glijdend gemiddelde (moving average) van de audio_data
# np.ones(window_size)/window_size maakt een gemiddelde-filter
# np.convolve voert de convolutie uit (filter toepassen)
# mode='valid' betekent: alleen volledige overlappingen gebruiken
# resultaat wordt opgeslagen in zowel 'data' als 'rolling_mean'

audio = audio_data.astype(np.float32) / 32768.0
# Zet audio_data om naar floats en normaliseert tussen -1 en 1
# 32768 komt van 16-bit audio (bereik -32768 tot 32767)

squared = audio**2
# Kwadrateert het signaal → nodig voor energie/RMS berekening

rolling = np.convolve(squared, np.ones(window_size)/window_size, mode='valid')
# Berekent het glijdend gemiddelde van het gekwadrateerde signaal

rms = np.sqrt(rolling)
# Neemt de wortel van het gemiddelde → RMS (Root Mean Square)
# Dit geeft de effectieve geluidssterkte

df = pd.DataFrame(rms, columns=["rms"])
# Zet de RMS-waarden in een nieuwe DataFrame met kolomnaam "rms"

# Remove NaNs safely
df = df.replace([np.inf, -np.inf], np.nan).dropna()
# Vervangt oneindige waarden (inf) door NaN en verwijdert deze rijen

# Convert to dBFS
df["dB"] = 20 * np.log10(df["rms"] + 1e-12)
# Zet RMS om naar decibel Full Scale (dBFS)
# +1e-12 voorkomt log(0), wat niet gedefinieerd is

df["time"] = np.arange(len(df)) / sampling_rate
# Maakt een tijd-as:
# np.arange(len(df)) = sample index
# delen door sampling_rate → tijd in seconden

df.plot(x = "time", y = "dB")
# Maakt een grafiek met tijd op de x-as en dB op de y-as

plt.ylabel("dBFS")
# Label voor de y-as

plt.show()
# Toont de grafiek

In [ ]:
import numpy as np              # Voor numerieke berekeningen (arrays, wiskunde)
import pandas as pd            # Voor werken met tabellen (DataFrames)
import matplotlib.pyplot as plt  # Voor het maken van grafieken

pd.set_option('display.max_rows', None)  
# Zorgt dat alle rijen zichtbaar zijn bij printen van een DataFrame

window_size = int(0.02 * sampling_rate)  
# Bepaalt de grootte van het venster (20 ms)
# sampling_rate = aantal samples per seconde
# Dus 0.02 * sampling_rate = aantal samples in 20 milliseconden

audio = audio_data.astype(np.float32) / 32768.0  
# Zet audio_data om naar floats en normaliseert tussen -1 en 1
# 32768 komt van 16-bit audio (bereik -32768 tot 32767)

squared = audio**2  
# Kwadrateert het signaal → nodig om energie/RMS te berekenen

rolling = np.convolve(squared, np.ones(window_size)/window_size, mode='valid')  
# Berekent het glijdend gemiddelde van het gekwadrateerde signaal
# np.ones(window_size)/window_size maakt een gemiddelde-filter
# mode='valid' gebruikt alleen volledige overlappingen

rms = np.sqrt(rolling)  
# Neemt de wortel → RMS (Root Mean Square)
# Dit geeft een maat voor de effectieve geluidssterkte

df = pd.DataFrame(rms, columns=["rms"])  
# Zet de RMS-waarden in een DataFrame met kolomnaam "rms"

df = df.replace([np.inf, -np.inf], np.nan).dropna()  
# Vervangt oneindige waarden (inf) door NaN en verwijdert deze rijen

df["dB"] = 20 * np.log10(df["rms"] + 1e-12)  
# Zet RMS om naar decibel Full Scale (dBFS)
# +1e-12 voorkomt log(0), wat niet gedefinieerd is

df["time"] = np.arange(len(df)) / sampling_rate  
# Maakt een tijd-as:
# np.arange(len(df)) = sample index
# delen door sampling_rate → tijd in seconden

# Plot
df.plot(x="time", y="dB")  
# Maakt een grafiek van tijd vs geluidsniveau (dBFS)

plt.ylabel("dBFS")  
# Label voor de y-as

plt.xlabel("Time (s)")  
# Label voor de x-as

plt.show()  
# Toont de grafiek

# --- NAGALMTIJD ---
Lmax = df["dB"].max()  
# Bepaalt het maximale geluidsniveau (piek in dB)

idx_max = df["dB"].idxmax()  
# Geeft de index (tijdstip) waar deze piek voorkomt

after_peak = df.loc[idx_max:]  
# Selecteert alleen het deel van de data ná de piek
# Dit is belangrijk omdat nagalm alleen daarna plaatsvindt

L1 = Lmax - 5  
# Eerste referentieniveau: 5 dB onder de piek

L2 = Lmax - 35  
# Tweede referentieniveau: 35 dB onder de piek

t1 = after_peak.loc[after_peak["dB"] <= L1, "time"].iloc[0]  
# Zoekt het eerste tijdstip waarop het geluid onder (Lmax - 5 dB) komt

t2 = after_peak.loc[after_peak["dB"] <= L2, "time"].iloc[0]  
# Zoekt het eerste tijdstip waarop het geluid onder (Lmax - 35 dB) komt

RT30 = t2 - t1  
# Tijd die het kost om 30 dB te dalen (van -5 dB tot -35 dB)

RT60 = 2 * RT30  
# Extrapoleert naar 60 dB verval → officiële nagalmtijd

print(f"Nagalmtijd (RT60): {RT60:.2f} s")  
# Print de berekende nagalmtijd in seconden

In [ ]:
import numpy as np              # Voor numerieke berekeningen (arrays, wiskunde)
import pandas as pd            # Voor werken met tabellen (DataFrames)
import matplotlib.pyplot as plt  # Voor het maken van grafieken

pd.set_option('display.max_rows', None)  
# Zorgt dat alle rijen zichtbaar zijn bij het printen van een DataFrame

df = pd.DataFrame(audio_data)  
# Zet de ruwe audio_data om in een DataFrame (tabelvorm)

window_size = int(0.02 * sampling_rate)  
# Bepaalt de grootte van het venster (20 ms)
# sampling_rate = aantal samples per seconde
# Dus 0.02 * sampling_rate = aantal samples in 20 milliseconden

audio = audio_data.astype(np.float32) / 32768.0  
# Zet audio om naar floats en normaliseert tussen -1 en 1
# 32768 komt van 16-bit audio (range -32768 tot 32767)

squared = audio**2  
# Kwadrateert het signaal → nodig voor energie/RMS berekening

rolling = np.convolve(squared, np.ones(window_size)/window_size, mode='valid')  
# Berekent het glijdend gemiddelde van het gekwadrateerde signaal
# np.ones(window_size)/window_size maakt een gemiddelde-filter
# mode='valid' gebruikt alleen volledige overlappingen

rms = np.sqrt(rolling)  
# Neemt de wortel → RMS (Root Mean Square)
# Dit geeft de effectieve geluidssterkte

df = pd.DataFrame(rms, columns=["rms"])  
# Zet de RMS-waarden in een nieuwe DataFrame met kolomnaam "rms"

df = df.replace([np.inf, -np.inf], np.nan).dropna()  
# Vervangt oneindige waarden (inf) door NaN en verwijdert deze rijen

# dBFS
df["dB"] = 20 * np.log10(df["rms"] + 1e-12)  
# Zet RMS om naar dBFS (decibel Full Scale)
# +1e-12 voorkomt log(0), wat niet gedefinieerd is

# tijd
df["time"] = np.arange(len(df)) / sampling_rate  
# Maakt een tijd-as:
# np.arange(len(df)) = sample index
# delen door sampling_rate → tijd in seconden

# kalibratie
offset = 120   # bepaal deze met referentiemeting
# Offset is een vaste correctie om van dBFS naar dB SPL te gaan
# Deze waarde moet je bepalen met een echte referentiemeting

df["dB_SPL"] = df["dB"] + offset  
# Zet dBFS om naar dB SPL door de offset op te tellen

# plot
df.plot(x="time", y="dB_SPL")  
# Maakt een grafiek met tijd op de x-as en dB SPL op de y-as

plt.ylabel("dB SPL")  
# Label voor de y-as

plt.xlabel("tijd (s)")  
# Label voor de x-as

plt.show()  
# Toont de grafiek

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

#clear old figures
#plt.close("all")
# (uitgecommentarieerd) zou alle bestaande grafieken sluiten als je meerdere plots maakt

#load CSV
df = pd.read_csv("Book2.csv", header=None)
# Leest het CSV-bestand in als een DataFrame
# header=None betekent dat de eerste rij geen kolomnamen is, maar data

#create figure
fig, ax = plt.subplots(figsize=(12, 8))
# Maakt een nieuwe figuur en as (grafiek)
# figsize bepaalt de grootte van de grafiek (breedte, hoogte)

#plot heatmap
#im = ax.imshow(df.values, cmap="inferno", aspect="auto", origin="lower")
# (oude versie, uitgecommentarieerd)

im = ax.imshow(df.values, cmap="inferno", aspect="auto", origin="lower", vmin=0, vmax=3)
# Maakt een heatmap van de data
# df.values = matrix van getallen uit de DataFrame
# cmap="inferno" = kleurenschaal (donker → licht)
# aspect="auto" = past de schaal automatisch aan
# origin="lower" = (0,0) begint linksonder
# vmin en vmax bepalen de schaal van de kleuren (hier van 0 tot 3)

#auto x/y labels
ax.set_xticks(range(df.shape[1]))
# Zet ticks op de x-as (voor elke kolom)

ax.set_yticks(range(df.shape[0]))
# Zet ticks op de y-as (voor elke rij)

ax.set_xticklabels([f"x{i+1}" for i in range(df.shape[1])])
# Geeft labels aan de x-as: x1, x2, x3, ...

ax.set_yticklabels([f"y{i+1}" for i in range(df.shape[0])])
# Geeft labels aan de y-as: y1, y2, y3, ...

#overlay numbers
for i in range(df.shape[0]):
    for j in range(df.shape[1]):
        # white text for dark cells, black for light cells
        color = "white" if df.iloc[i, j] > df.values.max() / 2 else "black"
        # Bepaalt tekstkleur afhankelijk van waarde (contrast)

        #ax.text(j, i, round(df.iloc[i, j], 2), ha="center", va="center", color=color)
        # (uitgecommentarieerd) zou de waarde in elke cel tonen als tekst

#attach colorbar
#fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
# (oude versie, uitgecommentarieerd)

# attach colorbar
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
# Voegt een kleurenschaal (colorbar) toe naast de heatmap
# fraction en pad bepalen grootte en afstand

cbar.set_label("Nagalmtijd(s)")
# Label voor de colorbar (hier: nagalmtijd in seconden)

#show
plt.title('Heatmap kerk VPGE')
# Titel van de grafiek

plt.xlabel('x-positie')
# Label voor de x-as

plt.ylabel('y-positie')
# Label voor de y-as

plt.show()
# Toont de grafiek